In [1]:
import os
import glob
import numpy as np
from google import genai
from google.genai import types

CHUNK_SIZE = 250
API_KEY = os.environ.get("GOOGLE_API_KEY")
PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT")

In [2]:
vertex_client = genai.Client(
    vertexai=True, 
    project=PROJECT,
    location="us-central1"
)
print("✅ Vertex Client loaded (for Embeddings)")

personal_client = genai.Client(api_key=API_KEY)
print("✅ Personal Client loaded (for Generation)")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✅ Vertex Client loaded (for Embeddings)
✅ Personal Client loaded (for Generation)


In [16]:
def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

In [17]:
from chunk_compare import chunk

corpus_chunks = []

for filepath in glob.glob("corpus/*.txt"):
    filename = os.path.basename(filepath)
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
        
    chunks = chunk(text, CHUNK_SIZE)
    
    for c in chunks:
        corpus_chunks.append({
            "source": filename,
            "text": c
        })

print(f"Total chunks created: {len(corpus_chunks)}")

Total chunks created: 271


In [19]:
def embed_document(text):
    r = vertex_client.models.embed_content(
        model="gemini-embedding-001",
        contents=text,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"),
    )
    return np.array(r.embeddings[0].values)

print("Embedding chunks... (this might take a minute)")
for doc in corpus_chunks:
    doc["embedding"] = embed_document(doc["text"])
    
print("Indexing complete.")

Embedding chunks... (this might take a minute)
Indexing complete.


In [20]:
def retrieve(query, k=3):
    r = vertex_client.models.embed_content(
        model="gemini-embedding-001",
        contents=query,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
    )
    query_vec = np.array(r.embeddings[0].values)
    
    scores = []
    for doc in corpus_chunks:
        score = cosine(query_vec, doc["embedding"])
        scores.append((score, doc))
        
    scores.sort(key=lambda x: x[0], reverse=True)
    
    return [doc for score, doc in scores[:k]]

In [31]:
rag_schema = types.Schema(
    type=types.Type.OBJECT,
    properties={
        "answer_found_in_context": types.Schema(
            type=types.Type.BOOLEAN,
            description="True if the context contains the answer, False if missing."
        ),
        "answer": types.Schema(
            type=types.Type.STRING,
            description="The answer to the user's question, or a refusal if missing."
        ),
        "supporting_quote": types.Schema(
            type=types.Type.STRING,
            description="An exact quote from the context that proves the answer. Empty if no answer."
        ),
        "source_document": types.Schema(
            type=types.Type.STRING,
            description="The source filename the quote came from. Empty if no answer."
        ),
    },
    required=["answer_found_in_context", "answer", "supporting_quote", "source_document"]
)

def ask_system(query):
    retrieved = retrieve(query, k=3)
    
    context_str = ""
    for i, doc in enumerate(retrieved):
        context_str += f"\n--- Source: {doc['source']} ---\n{doc['text'].strip()}\n"
        
    prompt = f"""
    You are an assistant answering questions for a Beloit College community member.
    Use ONLY the provided context below. 
    If the context does not contain the answer, set answer_found_in_context to False and refuse to answer.
    
    Context:
    {context_str}
    
    Question: {query}
    """
    
    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = personal_client.models.generate_content(
                model="gemini-3.8-flash",
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=rag_schema,
                    temperature=0.0
                )
            )
            return response.text
        except Exception as e:
            error_msg = str(e)
            if "503" in error_msg or "429" in error_msg:
                print(f"   [!] API busy (Attempt {attempt+1}/{max_retries}). Waiting 10s...")
                import time
                time.sleep(10)
            else:
                print(f"   [x] Fatal API Error: {error_msg[:100]}")
                return "{}"
                
    return "{}"

In [34]:
eval_set = [
    {
        "question": "What is the minimum number of units a student must be registered for to be considered full-time?",
        "expected_source": "catalog-adding-dropping-withdrawing.txt",
        "expected_fact": "At least 3.0 units. Three-quarter time is at least 2.25 units and half time is at least 1.50 units. The normal course load is 4–4.25 units."
    },
    {
        "question": "Which students are exempt from the requirement to live in College housing during their first six credit terms?",
        "expected_source": "handbook-housing-living-on-campus.txt",
        "expected_fact": "Married students, military veterans with two years of active duty, students with dependent children, and commuting students who live with their parents or legal guardians within a 30-minute commute. Any other exception needs prior approval of the Director of Residence Life."
    },
]

In [35]:
import json
import time

retrieval_hits = 0
grounded_answers = 0
correct_refusals = 0

total_answerable = 0
total_unanswerable = 0

print("Running Evaluation...\n" + "="*50)

for i, item in enumerate(eval_set):
    q = item["question"]
    expected_src = item["expected_source"]
    
    print(f"Question {i+1}: {q}")
    
    retrieved_docs = retrieve(q, k=3)
    retrieved_sources = [doc["source"] for doc in retrieved_docs]
    
    if expected_src is not None:
        total_answerable += 1
        if expected_src in retrieved_sources:
            retrieval_hits += 1
            print(f" [✓] Retrieval: Found '{expected_src}' in top 3")
        else:
            print(f" [x] Retrieval: Missed. Got {retrieved_sources}")
    else:
        total_unanswerable += 1

    response_json = ask_system(q)
    
    try:
        result = json.loads(response_json)
    except json.JSONDecodeError:
        print(" [x] Generation: Failed to parse JSON.")
        print("-" * 50)
        continue

    if expected_src is None:
        if result.get("answer_found_in_context") == False:
            correct_refusals += 1
            print(" [✓] Generation: Correctly refused to answer.")
        else:
            print(f" [x] Generation: Hallucinated an answer! \n     Source claimed: {result.get('source_document')}")
            
    else:
        if result.get("answer_found_in_context") == True and result.get("source_document") == expected_src:
            
            quote = result.get("supporting_quote", "")
            
            source_text = ""
            for doc in retrieved_docs:
                if doc["source"] == expected_src:
                    source_text = doc["text"]
                    break
            
            clean_quote = " ".join(quote.split())
            clean_source = " ".join(source_text.split())
            
            if clean_quote and clean_quote in clean_source:
                grounded_answers += 1
                print(" [✓] Generation: Answer grounded. Quote verified in source text.")
            else:
                print(f" [x] Generation: Ungrounded. Quote not found in '{expected_src}'.")
                print(f"     Quote generated: {quote}")
        else:
            print(f" [x] Generation: Failed to cite the correct source or claimed it couldn't find the answer.")
            
    print("-" * 50)
    time.sleep(4)
    

print("\n=== FINAL METRICS ===")
if total_answerable > 0:
    print(f"Retrieval Hit Rate: {retrieval_hits} / {total_answerable} ({retrieval_hits/total_answerable*100:.1f}%)")
    print(f"Grounding Accuracy: {grounded_answers} / {total_answerable} ({grounded_answers/total_answerable*100:.1f}%)")
if total_unanswerable > 0:
    print(f"Refusal Accuracy:   {correct_refusals} / {total_unanswerable} ({correct_refusals/total_unanswerable*100:.1f}%)")

Running Evaluation...
Question 1: What is the minimum number of units a student must be registered for to be considered full-time?
 [✓] Retrieval: Found 'catalog-adding-dropping-withdrawing.txt' in top 3
   [!] API busy (Attempt 1/3). Waiting 10s...
   [!] API busy (Attempt 2/3). Waiting 10s...
   [!] API busy (Attempt 3/3). Waiting 10s...
 [x] Generation: Failed to cite the correct source or claimed it couldn't find the answer.
--------------------------------------------------
Question 2: Which students are exempt from the requirement to live in College housing during their first six credit terms?
 [✓] Retrieval: Found 'handbook-housing-living-on-campus.txt' in top 3
   [!] API busy (Attempt 1/3). Waiting 10s...
   [!] API busy (Attempt 2/3). Waiting 10s...
   [!] API busy (Attempt 3/3). Waiting 10s...
 [x] Generation: Failed to cite the correct source or claimed it couldn't find the answer.
--------------------------------------------------

=== FINAL METRICS ===
Retrieval Hit Rate: